# Day 6 Practice: Tree-Based Models & Regularization

**Role:** Competitive Kaggle Grandmaster

Today, we benchmark the heavy hitters: LightGBM and XGBoost. You will build cross-validation pipelines, map overfitting curves, and learn to tame complex trees using stochastic regularization.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
import sys
import os

# Setup validation
try:
    import ipytest
except ImportError:
    !pip install -q ipytest
    import ipytest

if not os.path.exists('validate_day6.py'):
    print('⚠️ validate_day6.py not found!')
else:
    import validate_day6

ipytest.autoconfig()


## 1. Setup Synthetic Dataset
Generating a dataset with missing values and mixed types (categoricals).


In [ ]:
np.random.seed(42)
n_rows = 5000
df = pd.DataFrame({
    'f1': np.random.randn(n_rows),
    'f2': np.random.randn(n_rows),
    'f3': np.random.choice(['A', 'B', 'C', np.nan], n_rows),
    'Target': np.random.randint(0, 2, n_rows)
})
df['f3'] = df['f3'].astype('category')
X = df.drop(columns='Target')
y = df['Target']


## Challenge 1: LightGBM OOF Loop
**Task:** Train a baseline `LGBMClassifier` inside a 5-fold CV loop. Track the predictions in `lgbm_oof_preds`.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day6.test_task_1()
except NameError: pass


## Challenge 2: XGBoost Baseline
**Task:** Re-run the CV loop using an `XGBClassifier`. Use the same KFold indices as Challenge 1. Store in `xgb_oof_preds`.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day6.test_task_2()
except NameError: pass


## Challenge 3: Overfitting Curve
**Task:** Iterate through depths [2, 5, 10, 20, 50]. Train LGBM and record log-loss for both Train and Val sets. Plot to see where they diverge.


In [ ]:
# YOUR CODE HERE
# train_scores = []; val_scores = []


In [ ]:
%%ipytest -qq
try:
    validate_day6.test_task_3()
except NameError: pass


## Challenge 4: Stochastic Regularization
**Task:** Apply `subsample=0.8` and `colsample_bytree=0.8` to your best performing model. Check if the validation score (`reg_val_score`) improves compared to the overfitted baseline.


In [ ]:
# YOUR CODE HERE


In [ ]:
%%ipytest -qq
try:
    validate_day6.test_task_4()
except NameError: pass


<details>
<summary style='color: #d35400; font-weight: bold; cursor: pointer;'>🏆 Click for Grandmaster Optimized Solutions</summary>

```python
# 1 & 2. OOF Baselines
skf = StratifiedKFold(n_splits=5)
lgbm_oof_preds = np.zeros(len(df))
xgb_oof_preds = np.zeros(len(df))
for tr, val in skf.split(X, y):
    lgbm_oof_preds[val] = lgb.LGBMClassifier().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[val])[:,1]
    xgb_oof_preds[val] = xgb.XGBClassifier().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[val])[:,1]

# 3. Overfitting Curve
train_scores, val_scores = [], []
for d in [2, 5, 10, 20, 50]:
    model = lgb.LGBMClassifier(max_depth=d).fit(X.iloc[tr], y.iloc[tr])
    train_scores.append(log_loss(y.iloc[tr], model.predict_proba(X.iloc[tr])[:,1]))
    val_scores.append(log_loss(y.iloc[val], model.predict_proba(X.iloc[val])[:,1]))

# 4. Stochastic Regularization
reg_model = lgb.LGBMClassifier(subsample=0.8, colsample_bytree=0.8, max_depth=50).fit(X.iloc[tr], y.iloc[tr])
reg_val_score = log_loss(y.iloc[val], reg_model.predict_proba(X.iloc[val])[:,1])
```
</details>
